In [ ]:
import os
from dotenv import load_dotenv
from pycelonis import get_celonis

import requests
import pandas as pd

load_dotenv()  # Load variables from .env

In [ ]:
# Credentials loaded from .env
SYMBIO_BASE_URL = os.environ["SYMBIO_BASE_URL"]
SYMBIO_STORAGE_COLLECTION = os.environ["SYMBIO_STORAGE_COLLECTION"]
SYMBIO_TENANT = os.environ["SYMBIO_TENANT"]
TOKEN = os.environ["SYMBIO_AUTH_TOKEN"]

url = (
    f"{SYMBIO_BASE_URL}/{SYMBIO_STORAGE_COLLECTION}/{SYMBIO_TENANT}"
    f"/_api/rest/facets/processes/views/list/elements"
)

headers = {
    "symbio-auth-token": TOKEN,
    "api-version": "2.0",
    "Accept": "application/json",
}

# Now we extract the name of all processes
response = requests.get(url, headers=headers)
response.raise_for_status()

views = response.json()

In [ ]:
'''
users = {}
for element in views['elements']:
    authors = element.get('related').get('author')
    for author_aux in authors:
        users[author_aux.get('id')] = author_aux.get('expandUri')
    if element.get('related').get('owner') is not None:
        for owner in element.get('related').get('owner'):
            users[owner.get('id')] = owner.get('expandUri')
'''         
            
users = {}
for element in views['elements']:
    authors = element.get('related').get('author')
    if authors is not None:
        for author_aux in authors:
            users[author_aux.get('id')] = author_aux.get('expandUri')
    owners = element.get('related').get('owner')
    if owners is not None:
        for owner in owners:
            users[owner.get('id')] = owner.get('expandUri')
    additional_authors = element.get('related').get('secondaryAuthors')
    if additional_authors is not None:
        for additional_author in additional_authors:
            users[additional_author.get('id')] = additional_author.get('expandUri')

In [ ]:
final_user = []
for key, value in users.items():
    response_u = requests.get(value, headers=headers)
    response_u.raise_for_status()
    views_u = response_u.json()
 
    for aux_user in views_u['elements']:
        if aux_user['type'] == "authToken":
            continue
 
        final_user.append(
            {
                'id': aux_user['id'],
                'name': aux_user['attributes']['firstName']['127'],
                'last name': aux_user['attributes']['lastName']['127'],
                'email': aux_user['attributes']['profileEmail']['127']
 
            }
        )

In [ ]:
userDataframe = pd.DataFrame(final_user)

In [ ]:
celonis_team_url = os.environ["CELONIS_TEAM_URL"]
celonis_api_token = os.environ["CELONIS_API_TOKEN"]

celonis = get_celonis(
    base_url=celonis_team_url,
    api_token=celonis_api_token,
    key_type="APP_KEY",
)

data_pool_id = os.environ["CELONIS_DATA_POOL_ID"]
data_pool = celonis.data_integration.get_data_pool(data_pool_id)

In [ ]:
# Push to celonis
data_pool.create_table(df=userDataframe, table_name="CPMUsers", drop_if_exists=True,force=True)

# Process folders (Dropdown options)

Servicio para obtener carpetas de procesos (`ProcessGroup`) desde la API v2 de Symbio.

In [ ]:
from symbio_process_folders import get_process_folder_options, SymbioProcessFoldersError

STORAGE_COLLECTION = os.environ["SYMBIO_STORAGE_COLLECTION"]
TENANT = os.environ["SYMBIO_TENANT"]
BASE_URL = os.environ["SYMBIO_BASE_URL"]
TOKEN = os.environ["SYMBIO_AUTH_TOKEN"]

try:
    # recursive=True expande /v2/data/elements/{id} para obtener todas las carpetas del árbol
    process_folders = get_process_folder_options(
        storagecollection=STORAGE_COLLECTION,
        tenant=TENANT,
        symbio_auth_token=TOKEN,
        base_url=BASE_URL,
        recursive=True,
        folder_types={"subCategory", "processGroup", "category", "mainProcess"},
    )
except SymbioProcessFoldersError as error:
    print(f"Error al obtener carpetas: {error}")
    process_folders = []

print(f"Total carpetas encontradas: {len(process_folders)}")
process_folders[:10]

In [ ]:
import pandas as pd

process_folders_df = pd.DataFrame(process_folders)
process_folders_df.sort_values("name", na_position="last")